# UniAssist — Week 3: Morphology, Stemming, Lemmatization and WordNet
Use the original UMT corpus to compare stemming algorithms, POS-aware lemmatization and semantic relations. We evaluate outputs on examples rather than presuming that any single algorithm is correct for every word.


In [1]:
from pathlib import Path
import json, re, statistics
from collections import Counter
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "corpus").exists():
    ROOT = ROOT.parent
assert (ROOT / "corpus").exists(), "Run notebook from UniAssist_UMT directory"
metadata = json.loads((ROOT / "metadata.json").read_text(encoding="utf-8"))
records = metadata["documents"]
texts = [(ROOT / "corpus" / row["file"]).read_text(encoding="utf-8") for row in records]
print(len(texts), "documents loaded")


20 documents loaded


In [2]:
import nltk, spacy
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet as wn
from nltk import pos_tag
for dataset in ("wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"):
    nltk.download(dataset, quiet=True)
porter = PorterStemmer()
snowball = SnowballStemmer("english")
lancaster = LancasterStemmer()
lemmatizer = WordNetLemmatizer()
nlp = spacy.blank("en")


## Task A — Compare three stemmers on vocabulary from UMT documents


In [3]:
words = ["studies", "studying", "universities", "student", "students",
         "admissions", "admitted", "evaluation", "evaluated", "applications",
         "faculties", "scholarships", "architectural", "mobility", "programmes"]
stem_table = pd.DataFrame([{
    "word": word, "Porter": porter.stem(word),
    "Snowball": snowball.stem(word), "Lancaster": lancaster.stem(word)
} for word in words])
display(stem_table)
print("Porter distinct stems:", stem_table["Porter"].nunique())
print("Snowball distinct stems:", stem_table["Snowball"].nunique())
print("Lancaster distinct stems:", stem_table["Lancaster"].nunique())


,word,Porter,Snowball,Lancaster
0,studies,studi,studi,study
1,studying,studi,studi,study
2,universities,univers,univers,univers
3,student,student,student,stud
4,students,student,student,stud
5,admissions,admiss,admiss,admit
6,admitted,admit,admit,admit
7,evaluation,evalu,evalu,evalu
8,evaluated,evalu,evalu,evalu
9,applications,applic,applic,apply


Porter distinct stems: 12
Snowball distinct stems: 12
Lancaster distinct stems: 11


## Task B — Corpus-level vocabulary reduction
Stemming is a string normalization technique, not guaranteed grammatical lemmatization. Compare unique terms only after lowercasing and filtering alphabetic tokens.


In [4]:
words_in_corpus = [token.text.lower() for text in texts
                   for token in nlp.make_doc(text) if token.is_alpha]
rows = []
for name, transform in [("unchanged", lambda s:s), ("Porter", porter.stem),
                        ("Snowball", snowball.stem), ("Lancaster", lancaster.stem)]:
    forms = [transform(word) for word in words_in_corpus]
    rows.append({"method": name, "tokens": len(forms), "types": len(set(forms)),
                 "TTR": round(len(set(forms))/max(1, len(forms)), 4)})
display(pd.DataFrame(rows))


,method,tokens,types,TTR
0,unchanged,5098,1127,0.2211
1,Porter,5098,842,0.1652
2,Snowball,5098,840,0.1648
3,Lancaster,5098,778,0.1526


## Task C — Lemmatization without POS versus POS-aware WordNet lemmatization


In [5]:
def wn_pos(tag):
    # Penn Treebank tag initials: adjective, verb, noun, adverb.
    return {"J":wn.ADJ, "V":wn.VERB, "N":wn.NOUN, "R":wn.ADV}.get(tag[:1], wn.NOUN)

samples = ["Students were studying admission requirements carefully.",
           "The faculties offered different programmes and scholarships.",
           "Visitors asked about better learning facilities and transferred credits."]
lemma_rows = []
for sentence in samples:
    tokens = nltk.word_tokenize(sentence, preserve_line=True)
    tagged = pos_tag(tokens)
    for word, tag in tagged:
        if word.isalpha():
            lemma_rows.append({"sentence": sentence, "word": word, "POS": tag,
                "default_noun": lemmatizer.lemmatize(word.lower()),
                "POS_aware": lemmatizer.lemmatize(word.lower(), wn_pos(tag))})
display(pd.DataFrame(lemma_rows))


,sentence,word,POS,default_noun,POS_aware
0,Students were studying admission requirements ...,Students,NNS,student,student
1,Students were studying admission requirements ...,were,VBD,were,be
2,Students were studying admission requirements ...,studying,VBG,studying,study
3,Students were studying admission requirements ...,admission,NN,admission,admission
4,Students were studying admission requirements ...,requirements,NNS,requirement,requirement
5,Students were studying admission requirements ...,carefully,RB,carefully,carefully
6,The faculties offered different programmes and...,The,DT,the,the
7,The faculties offered different programmes and...,faculties,NNS,faculty,faculty
8,The faculties offered different programmes and...,offered,VBD,offered,offer
9,The faculties offered different programmes and...,different,JJ,different,different


## Task D — spaCy lemmatization on actual source-based text
This requires the trained English model. Compare it with the WordNet POS-aware results and note errors for UMT proper names, acronyms, and domain vocabulary.


In [6]:
try:
    trained = spacy.load("en_core_web_sm")
except OSError:
    print("Missing spaCy language model. Run: python -m spacy download en_core_web_sm")
else:
    sample_doc = trained(texts[0][:1300])
    display(pd.DataFrame([{"token": tok.text, "lemma": tok.lemma_, "coarse_POS": tok.pos_,
                           "fine_POS": tok.tag_} for tok in sample_doc if tok.is_alpha]).head(55))


,token,lemma,coarse_POS,fine_POS
0,University,University,PROPN,NNP
1,Metropolitan,Metropolitan,PROPN,NNP
2,Tirana,Tirana,PROPN,NNP
3,Institution,Institution,PROPN,NNP
4,Overview,Overview,PROPN,NNP
5,University,University,PROPN,NNP
6,Metropolitan,Metropolitan,PROPN,NNP
7,Tirana,Tirana,PROPN,NNP
8,UMT,UMT,PROPN,NNP
9,is,be,AUX,VBZ


## Task E — WordNet senses, hypernyms, and semantic similarity
WordNet describes general English, not UMT's internal policies. The first WordNet sense is not automatically the correct sense in a university context.


In [7]:
terms = ["university", "student", "course", "admission", "library", "faculty"]
for term in terms:
    senses = wn.synsets(term, pos=wn.NOUN)
    print("\n", term, "senses:", len(senses))
    for syn in senses[:2]:
        print(" ", syn.name(), "-", syn.definition())
        print("   hypernyms:", [h.name() for h in syn.hypernyms()[:3]])

pairs = [("student", "scholar"), ("university", "college"),
         ("course", "class"), ("library", "school")]
for left, right in pairs:
    a = wn.synsets(left, pos=wn.NOUN)
    b = wn.synsets(right, pos=wn.NOUN)
    value = a[0].wup_similarity(b[0]) if a and b else None
    print(left, right, "first-sense Wu-Palmer similarity:", value)



 university senses: 3
  university.n.01 - the body of faculty and students at a university
   hypernyms: ['body.n.02']
  university.n.02 - establishment where a seat of higher learning is housed, including administrative and living quarters as well as facilities for research and teaching
   hypernyms: ['establishment.n.04']

 student senses: 2
  student.n.01 - a learner who is enrolled in an educational institution
   hypernyms: ['enrollee.n.01']
  scholar.n.01 - a learned person (especially in the humanities); someone who by long study has gained mastery in one or more disciplines
   hypernyms: ['intellectual.n.01']

 course senses: 9
  course.n.01 - education imparted in a series of lessons or meetings
   hypernyms: ['education.n.01']
  course.n.02 - a connected series of events or actions or developments
   hypernyms: ['series.n.01']

 admission senses: 4
  admission.n.01 - the act of admitting someone to enter
   hypernyms: ['entrance.n.03']
  admission.n.02 - an acknowledgment of

## Task F — Error analysis and implications for search


In [8]:
examples = ["studies", "universities", "admission", "programmes",
            "Erasmus", "UMT", "better", "running", "architecture"]
display(pd.DataFrame([{"input": w, "Porter": porter.stem(w),
                       "Snowball": snowball.stem(w),
                       "noun_lemma": lemmatizer.lemmatize(w.lower())} for w in examples]))


,input,Porter,Snowball,noun_lemma
0,studies,studi,studi,study
1,universities,univers,univers,university
2,admission,admiss,admiss,admission
3,programmes,programm,programm,programme
4,Erasmus,erasmu,erasmus,erasmus
5,UMT,umt,umt,umt
6,better,better,better,better
7,running,run,run,running
8,architecture,architectur,architectur,architecture


## Submit
1. Compare 15 words across Porter, Snowball and Lancaster.
2. Explain observed vocabulary reduction and at least three overstemming/understemming cases.
3. Contrast default WordNet, POS-aware WordNet and spaCy lemmas with examples.
4. Display at least six WordNet lookup results and four semantic comparisons.
5. State which normalization you would use for the eventual TF-IDF search and why, including limitations.
